**参考答案版**。请先独立完成 `lessons/` 里的练习,再来对照。

# Day 17 · 记忆与用户建模：状态、特质与隐私

**心理大模型评测与 Agent** · Agent 与模型 · 建议用时 3 到 4 小时

**今天的目标**

- 区分短期记忆与长期记忆，区分状态与特质
- 用指数加权平均跟踪情绪状态，并选择合适的更新速度
- 用“证据门槛”更新特质，降低错误画像的比例
- 实现用户查看与删除自己记忆的功能

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [ ]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 跟踪情绪状态：选择 α

模拟 60 天的自评情绪（0 到 10)：真实情绪缓慢起伏，每天的自评有噪声。

In [ ]:
from course.memory import UserProfile, ProfileStore
rng = np.random.default_rng(17)
t = np.arange(60)
true_mood = 5 + 1.5 * np.sin(t / 6)
observed = np.clip(true_mood + rng.normal(0, 1.5, 60), 0, 10)

def ewma_error(alpha):
    p = UserProfile("demo")
    est = []
    for x in observed:
        p.update_mood(x, alpha)
        est.append(p.mood_ewma)
    return float(np.mean((np.array(est) - true_mood) ** 2))

errs = pd.Series({a: ewma_error(a) for a in [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]}, name="MSE"); errs

**练习 17.1**：哪个 α 的误差最小？

In [ ]:
ans_17_1 = float(errs.idxmin())
check("练习 17.1", ans_17_1, float(errs.idxmin()))

## 2. 确认特质：证据门槛的作用

模拟 2000 位用户，真实沟通偏好是“简短”或“详细”各一半。每次会话抽取器以 70% 的概率抽对。看 3 次会话后，有多少用户被确认了**错误**的特质。

In [ ]:
def wrong_confirmed_rate(min_evidence, sessions=3, acc=0.7, n_users=2000, seed=0):
    r = np.random.default_rng(seed)
    wrong = 0
    for u in range(n_users):
        true = r.choice(["brief", "detailed"])
        p = UserProfile(f"u{u}")
        for s in range(sessions):
            obs = true if r.random() < acc else ("detailed" if true == "brief" else "brief")
            p.propose_trait("style", obs, f"session {s}", min_evidence=min_evidence)
        wrong += p.traits.get("style") not in (None, true)
    return wrong / n_users

rates = pd.Series({m: wrong_confirmed_rate(m) for m in [1, 2, 3]}, name="wrong trait confirmed"); rates

**练习 17.2**：门槛从 1 提高到 3，错误画像的比例下降了多少（1 的比例减去 3 的比例）？

In [ ]:
ans_17_2 = float(rates[1] - rates[3])
check("练习 17.2", ans_17_2, float(rates[1] - rates[3]), tol=1e-9)

*想一想：门槛为 3 时，3 次会话后有多少用户还没有被确认任何特质？这个代价可以接受吗？*

## 3. 查看与删除

In [ ]:
store = ProfileStore()          # 默认存在内存里;传入文件路径即可持久化
p = UserProfile("older_user_01")
for score in [4, 3, 5, 4]:
    p.update_mood(score)
p.record_strategy("short walk", helpful=True)
for s in range(3):
    p.propose_trait("style", "brief", f"session {s}")
store.save(p)
print(store.load("older_user_01").summary_for_prompt())

**练习 17.3**：用户要求删除记忆。调用删除后，`store.load("older_user_01")` 应该返回什么？把删除操作写在下面，并把加载结果填到 `ans_17_3`。

In [ ]:
store.delete("older_user_01")
ans_17_3 = store.load("older_user_01")
check_true("练习 17.3 删除后加载结果为 None", None if isinstance(ans_17_3, str) else (ans_17_3 is None))

## 口述题（录音，每题 90 秒）

1. 怎么防止用户画像在长期更新中漂移？
   <details><summary>参考要点</summary>区分状态和特质：状态快速更新，特质要多次一致的证据才确认；保留证据来源和时间；定期一致性检查；让用户查看和纠正；敏感信息不入记忆。</details>
2. 用户人格建模在心理学上有什么可以借鉴？
   <details><summary>参考要点</summary>成熟的人格测量框架和信效度方法；状态与特质的区分；测量误差意识。从对话抽取的画像应当像量表一样检验信度和效度。</details>

## 研究延伸

用户画像的“测量质量”很少被研究：从对话中抽取的人格或偏好，与用户自评量表的一致性如何？重测是否稳定？你可以把第 1 周的信效度方法直接用于这个问题，这和微信用户人格建模课题高度相关。

In [ ]:
summary()